# Phase 3 Results — Accuracy Push vs RBF (Closing Notebook)

**Benchmark:** `spy_phase1_random40_noiselow` (dirty, full fold) and the
corrected single-valued **`…_otm`** substrate (the operative verdict, per
ADR 0006). Generated by `scripts/generate_phase3_results_notebook.py`
(story 3D.2); numbers finalised in `docs/phase3_result_memo.md` (3D.3) and
[ADR 0009](../docs/decisions/0009_phase3_production_predictor_selection.md)
(3D.4).

## TL;DR

Phase 3 attacked the gap to the per-date RBF interpolation baseline along
three independent levers — coordinate representation (3A), a cross-attention
decoder (3B), and microstructure features (3C) — plus a data-correction
interlude (3X). **The acceptance bar (test MAE ≤ 0.95 × RBF, on its own) was
NOT met by any variant.** On the clean OTM substrate RBF (test MAE
**0.00613**) beats the best neural head (ANP point, **0.00987**, +61 %); the
calibrated production predictor is +90 %. Correcting the dirty call-put
duplicate confound (3X) *widened* RBF's lead rather than closing it, and the
`micro_v1` features (3C) *worsened* test MAE in all three heads. The
DeepSets→ANP architecture story survives (ANP beats DeepSets at every matched
head). Forward direction: **Phase 4 = RBF-prior hybrid / residual neural**.

Every table/figure below is read from a committed artifact at execution time;
no model is loaded and nothing is re-trained or re-scored.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# Resolve the repo root whether this notebook is executed from notebooks/
# or from the repo root.
REPO = Path.cwd()
if not (REPO / 'results').is_dir() and (REPO.parent / 'results').is_dir():
    REPO = REPO.parent

DIRTY = REPO / 'results' / '3' / 'spy_phase1_random40_noiselow'
OTM = REPO / 'results' / '3' / 'spy_phase1_random40_noiselow_otm'
RUNS = REPO / 'artifacts' / 'runs'
RESULTS_2D = REPO / 'results' / '2D'

print('REPO       =', REPO)
print('DIRTY ok   :', DIRTY.is_dir())
print('OTM ok     :', OTM.is_dir())
print('RUNS ok    :', RUNS.is_dir())


## 1. Headline ladder — clean OTM substrate (the operative verdict)

`3x_compare/comparison_wide.csv` is the matched dirty-vs-clean restatement:
each model family/head's full-fold dirty test MAE beside its clean-OTM test
MAE. The RBF floor on OTM is **0.00613**; the ratio column shows how much the
dirty call-put confound was flattering each model (RBF gained the most from
the correction, which is why its lead *widened*).

In [ ]:
wide = pd.read_csv(OTM / '3x_compare' / 'comparison_wide.csv')
wide = wide.sort_values('otm_test_mae').reset_index(drop=True)
rbf_otm = wide.loc[wide['family'] == 'rbf', 'otm_test_mae'].iloc[0]
wide['otm_vs_rbf_pct'] = 100.0 * (wide['otm_test_mae'] - rbf_otm) / rbf_otm
display(wide[['family', 'head', 'dirty_test_mae', 'otm_test_mae',
             'dirty_over_otm_ratio', 'otm_vs_rbf_pct']].round(4))
print(f'RBF-on-OTM test-MAE floor = {rbf_otm:.5f}')


## 2. Full model-family ladder (long form)

`3x_compare/comparison.csv` carries every family × head × substrate × split ×
metric cell with its provenance `source`. The pivot below is test-MAE per
family/head on each substrate.

In [ ]:
long = pd.read_csv(OTM / '3x_compare' / 'comparison.csv')
mae = long[(long['metric'] == 'mae') & (long['split'] == 'test')].copy()
pivot = mae.pivot_table(index=['family', 'head'], columns='data_substrate',
                        values='value')
display(pivot.round(5))


## 3. W10 / 3A — coordinate-representation ablation (Fourier vs raw)

Decoder-only retrain on the frozen 2D.7 encoder. Raw `(k, τ)` beats Fourier
features on full-fold test MAE; neither closes the gap to RBF. Source:
`3a_compare/comparison.csv`.

In [ ]:
a = pd.read_csv(DIRTY / '3a_compare' / 'comparison.csv')
display(a[a['metric'] == 'mae'].pivot_table(
    index='variant', columns='split', values='value').round(5))


## 4. W11 / W11.5 — calibrated ANP reliability layer (dirty vs OTM)

The end-to-end DeepSets+ANP with the Phase 2D calibration recipe. The
reliability layer (coverage ≈ 0.90, hi-conf MAE < no-abstention MAE) holds on
both substrates even though the point surface does not beat RBF. Sources:
`3b_anp/metrics_summary.csv` (dirty), `3x_anp/metrics_summary.csv` (OTM).

In [ ]:
b = pd.read_csv(DIRTY / '3b_anp' / 'metrics_summary.csv')
x = pd.read_csv(OTM / '3x_anp' / 'metrics_summary.csv')
print('=== 3B ANP calibrated (dirty) ===')
display(b)
print('=== 3X ANP calibrated (OTM) ===')
display(x)


In [ ]:
for label, d in (('3B (dirty)', DIRTY / '3b_anp'),
                  ('3X (OTM)', OTM / '3x_anp')):
    for fig in ('calibration_plot.png', 'abstention_curve.png'):
        p = d / fig
        print(f'{label}: {fig}')
        if p.exists():
            display(Image(filename=str(p)))
        else:
            print(f'  (missing: {p})')


## 5. W12 / 3C — `micro_v1` microstructure features (negative result)

Three-head ANP retrain on OTM with the 9-dim `micro_v1` context (ADR 0008).
Test MAE *regressed* in every head vs the matched 3X.9 minimal-feature
baseline — a clean negative. Read from the committed run manifests.

In [ ]:
import json

rows = []
baseline_3x9 = {'gaussian': 0.014404, 'quantile': 0.011752, 'point': 0.009871}
for head in ('gaussian', 'quantile', 'point'):
    m = json.loads((RUNS / '3C3' / head / 'manifest.json').read_text())
    rows.append({
        'head': head,
        'feature_set': m.get('feature_set'),
        'context_dim': m.get('context_dim'),
        'micro_v1_test_mae': round(float(m['test_mae_mu']), 6),
        '3x9_minimal_test_mae': baseline_3x9[head],
        'delta': round(float(m['test_mae_mu']) - baseline_3x9[head], 6),
    })
display(pd.DataFrame(rows))
print('All three heads worse than the minimal baseline -> micro_v1 not adopted.')


## 6. Training dynamics (representative curves)

Per-run training/validation loss curves. Filenames differ by run era
(`training_curve.csv` vs `training_curves.csv`); the cell handles both.

In [ ]:
import matplotlib.pyplot as plt

curve_files = {
    '3A raw': RUNS / '3A' / 'raw' / 'training_curves.csv',
    '3A fourier': RUNS / '3A' / 'fourier' / 'training_curves.csv',
    '3B4 point': RUNS / '3B4' / 'point_control' / 'training_curve.csv',
    '3X9 point': RUNS / '3X9' / 'point_control' / 'training_curve.csv',
    '3C3 point': RUNS / '3C3' / 'point' / 'training_curve.csv',
}
fig, ax = plt.subplots(figsize=(9, 5))
for label, path in curve_files.items():
    if not path.exists():
        print(f'(missing: {path})')
        continue
    c = pd.read_csv(path)
    ycol = next((col for col in ('val_loss', 'val_mae', 'valid_loss', 'loss')
                 if col in c.columns), c.columns[-1])
    xcol = 'epoch' if 'epoch' in c.columns else c.columns[0]
    ax.plot(c[xcol], c[ycol], marker='.', label=f'{label} ({ycol})')
ax.set_xlabel('epoch'); ax.set_ylabel('validation metric')
ax.set_title('Phase 3 training dynamics (representative runs)')
ax.legend(); ax.grid(alpha=0.3)
plt.show()


## 7. Acceptance-criteria map (roadmap §5)

| Criterion (roadmap §5) | Result | Evidence |
|---|---|---|
| Test MAE ≤ 0.95 × RBF on the benchmark, model on its own | **NOT met** — best OTM head +61 % vs RBF | `3x_compare/comparison_wide.csv` |
| No reliability regression (coverage within ±2 pp of 0.90) | Met — OTM coverage 0.9295 | `3x_anp/metrics_summary.csv` |
| Hi-conf MAE < no-abstention MAE | Met — OTM hi-conf 0.00835 < 0.01162 | `3x_anp/metrics_summary.csv` |
| RBF-as-prior / residual hybrids out of scope for Phase 3 | Held — reserved for Phase 4 | ADR 0004 / ADR 0009 |

The accuracy bar is the gating criterion and it is not met; the reliability
criteria are met. See `docs/phase3_result_memo.md` for the full mapping.

## 8. Phase 2D anchor (dirty substrate)

`results/2D/comparison_summary.csv` — the decision-layer baselines Phase 3
was measured against on the original dirty substrate.

In [ ]:
display(pd.read_csv(RESULTS_2D / 'comparison_summary.csv').round(5))


## 9. Open questions / Phase 4 framing

1. **Phase 4 = RBF-prior hybrid / residual neural model.** Let RBF carry the
   local interpolation it already wins; have the neural model learn the
   residual (sparse wings / extreme maturities) plus the calibrated
   reliability + abstention layer RBF lacks. Deployment answer (ADR 0004 /
   ADR 0009), not a retraction of the Phase 3 research question.
2. **`micro_v1` log-transform rescue** is reserved as an opt-in (ADR 0008
   Outcome) but not pursued — pure feature expansion already missed the bar.
3. **All-11-OTM-variant robustness study** remains deferred (no-overclaim
   guardrail: every clean-OTM number here is scoped to the matched
   `random40_noiselow_otm` substrate only).

## Reproducing this notebook

```bash
python3 scripts/generate_phase3_results_notebook.py
jupyter nbconvert --to notebook --execute \
    notebooks/06_phase3_results.ipynb --output /tmp/_3D4_check.ipynb
```

No new training, no new scoring — every numeric claim traces to a committed
artifact path.